In [2]:
!pip install cupy-cuda13x pandas tqdm scikit-learn 'spacy[transformers,lookups]' 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 MB 63.4 MB/s  0:00:00 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 214.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 77.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 98.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.2/33.2 MB 153.7 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.0/875.0 kB 195.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 193.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.5/98.5 MB 89.3 MB/s  0:00:01 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 93.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 120.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 162.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 182.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10

In [3]:
!python -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 40.9 MB/s  0:00:17:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 37.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [en-core-web-trf] [en-core-web-trf]
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')


In [4]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 56.0 MB/s  0:00:07:00:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


In [6]:
import pandas as pd
import spacy
from tqdm import tqdm
from spacy.tokens import DocBin
from sklearn.model_selection import train_test_split

In [7]:
df = pd.read_csv("hate_speech_clean_master.csv")
df.head()

,text,label,token_len
0,denial of normal the con be asked to comment o...,1,20
1,just by being able to tweet this insufferable ...,1,23
2,that is retarded you too cute to be single tha...,1,16
3,thought of a real badass mongol style declarat...,1,23
4,afro american basho,1,6


In [8]:
df["text"] = df["text"].astype("str")
df["label"] = df["label"].astype("int")
df = df.drop(columns=['token_len'])

In [9]:
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=108, stratify=df["label"]
)

dev_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=108, stratify=temp_df["label"]
)

In [10]:
def db_creator(df):
    db = DocBin()
    nlp = spacy.blank("en")

    for text, label in tqdm(df.values):
        doc = nlp.make_doc(text)
        if label == 1:
            doc.cats = {"Hateful": 1, "Not-Hateful": 0}
        elif label == 0:
            doc.cats = {"Hateful": 0, "Not-Hateful": 1}
        db.add(doc)

    return db

In [11]:
train_db = db_creator(train_df)
train_db.to_disk("docbins/train.spacy")

100%|██████████| 356912/356912 [01:06<00:00, 5336.43it/s]


In [12]:
dev_db = db_creator(dev_df)
dev_db.to_disk("docbins/dev.spacy")

100%|██████████| 44614/44614 [00:08<00:00, 5127.78it/s]


In [13]:
test_db = db_creator(test_df)
test_db.to_disk("docbins/test.spacy")

100%|██████████| 44615/44615 [00:08<00:00, 5107.80it/s]


Let's generate the config files for spacy

In [14]:
!python -m spacy init fill-config configs/base_config.cfg configs/static_vector.cfg
!python -m spacy init fill-config configs/base_config_gpu_bert.cfg configs/gpu_bert.cfg
!python -m spacy init fill-config configs/base_config_gpu_distilbert.cfg configs/gpu_distilbert.cfg
!python -m spacy init fill-config configs/base_config_gpu_roberta.cfg configs/gpu_roberta.cfg
!python -m spacy init fill-config configs/base_config_gpu_electra.cfg configs/gpu_electra.cfg

✔ Auto-filled config with all values
✔ Saved config
configs/static_vector.cfg
You can now add your data and train your pipeline:
python -m spacy train static_vector.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy
✔ Auto-filled config with all values
✔ Saved config
configs/gpu_bert.cfg
You can now add your data and train your pipeline:
python -m spacy train gpu_bert.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy
✔ Auto-filled config with all values
✔ Saved config
configs/gpu_distilbert.cfg
You can now add your data and train your pipeline:
python -m spacy train gpu_distilbert.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy
✔ Auto-filled config with all values
✔ Saved config
configs/gpu_roberta.cfg
You can now add your data and train your pipeline:
python -m spacy train gpu_roberta.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy
✔ Auto-filled config with all values
✔ Saved config
configs/gpu_electra.cfg
You can now add your data and train your pipeline:
p

Training the models

In [15]:
!time python -m spacy train configs/static_vector.cfg --paths.train docbins/train.spacy --paths.dev docbins/dev.spacy --output output/static_vector --gpu-id 0

✔ Created output directory: output/static_vector
ℹ Saving to output directory: output/static_vector
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'textcat']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS TEXTCAT  CATS_SCORE  SCORE 
---  ------  ------------  ------------  ----------  ------
  0       0          0.00          0.25       51.79    0.52
  0     200         20.72         30.06       45.13    0.45
  0     400        122.79         31.96       51.57    0.52
  0     600        702.09         27.38       47.86    0.48
  0     800        836.82         37.50       45.82    0.46
  0    1000        697.17         29.42       54.64    0.55
  0    1200       1613.25         28.59       52.12    0.52
  0    1400       1605.55         32.34       51.35    0.51
  0    1600       1129.29         33.02  

In [20]:
!time python -m spacy train configs/gpu_roberta.cfg --paths.train docbins/train.spacy --paths.dev docbins/dev.spacy --output output/roberta --gpu-id 0

ℹ Saving to output directory: output/roberta
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'textcat']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS TEXTCAT  CATS_SCORE  SCORE 
---  ------  -------------  ------------  ----------  ------
/venv/main/lib/python3.12/site-packages/thinc/util.py:395: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
/venv/main/lib/py

In [21]:
!time python -m spacy train configs/gpu_bert.cfg --paths.train docbins/train.spacy --paths.dev docbins/dev.spacy --output output/bert --gpu-id 0

ℹ Saving to output directory: output/bert
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'textcat']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS TEXTCAT  CATS_SCORE  SCORE 
---  ------  -------------  ------------  ----------  ------
/venv/main/lib/python3.12/site-packages/thinc/util.py:395: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
/venv/main/lib/python3.12/site-packages/thinc/util.py:395: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
  0       0           0.00          0.75       

In [22]:
!time python -m spacy train configs/gpu_distilbert.cfg --paths.train docbins/train.spacy --paths.dev docbins/dev.spacy --output output/distilbert --gpu-id 0

ℹ Saving to output directory: output/distilbert
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'textcat']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS TEXTCAT  CATS_SCORE  SCORE 
---  ------  -------------  ------------  ----------  ------
/venv/main/lib/python3.12/site-packages/thinc/util.py:395: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
/venv/main/lib/python3.12/site-packages/thinc/util.py:395: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
  0       0           0.00          0.75 

In [23]:
!time python -m spacy train configs/gpu_electra.cfg --paths.train docbins/train.spacy --paths.dev docbins/dev.spacy --output output/electra --gpu-id 0

ℹ Saving to output directory: output/electra
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'textcat']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS TEXTCAT  CATS_SCORE  SCORE 
---  ------  -------------  ------------  ----------  ------
/venv/main/lib/python3.12/site-packages/thinc/util.py:395: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
/venv/main/lib/python3.12/site-packages/thinc/util.py:395: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
  0       0           0.00          0.25    